# XLasso Simulation Experiments - Comprehensive Comparison

This notebook runs all simulation experiments in one place and displays results inline.
Experiments systematically evaluate the two key innovations of XLasso:

1. **Group constraint** - promotes grouped selection of correlated features
2. **Adaptive weighting** - penalty adjustment based on univariate significance

We compare four configurations to isolate the effect of each innovation:
- Baseline: ❌ no grouping, ❌ no adaptive weighting  (similar to standard Lasso)
- Adaptive only: ❌ no grouping, ✅ adaptive weighting  
- Grouping only: ✅ grouping, ❌ no adaptive weighting
- Full XLasso: ✅ grouping, ✅ adaptive weighting (complete algorithm)


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

from experiments.linear_experiment import LinearGaussianExperiment, run_all_linear_scenarios
from experiments.glm_experiment import GLMExperiment, run_all_glm_experiments
from experiments.nonlinear_experiment import NonlinearExperiment, run_all_nonlinear_experiments
from experiments import visualization

OUTPUT_DIR = '../experiments/results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('All imports completed.')

All imports completed.


## Part 1: Linear Gaussian Experiments

Systematic evaluation across different correlation structures:
- independent features
- AR(1) correlation
- block structure correlation
- block structure with only one true variable per block (tests false discovery)
- sign inconsistent (two highly correlated true variables with opposite signs)
- factor model
- high-dimensional block structure at different SNR levels


In [ ]:
# Run all linear scenarios (set n_repeats=5 for quick testing, 30 for full results)
N_REPEATS = 5  # Change to 30 for full publication-quality results

linear_results = run_all_linear_scenarios(
    output_dir=OUTPUT_DIR,
    n_repeats_override=N_REPEATS,
    save_plots=True
)

print('\n=== Linear Experiments Summary ===')
for scenario, df in linear_results.items():
    print(f'\n**{scenario}**')
    display(df[['f1', 'tpr', 'fpr', 'mse']].round(4))


Running scenario: independent
Starting experiment with 5 repetitions...
  Repeat 1/5 (seed=15795) [XLasso (baseline)]

### Linear Experiments Summary Table

In [ ]:
# Create a summary comparison table
summary_rows = []
for scenario, df in linear_results.items():
    for method in df.index:
        row = {
            'Scenario': scenario,
            'Method': method,
            'F1 (mean)': df.loc[method, ('f1', 'mean')],
            'F1 (std)': df.loc[method, ('f1', 'std')],
            'TPR': df.loc[method, ('tpr', 'mean')],
            'FPR': df.loc[method, ('fpr', 'mean')],
            'MSE': df.loc[method, ('mse', 'mean')],
        }
        summary_rows.append(row)

linear_summary_df = pd.DataFrame(summary_rows)
linear_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'linear_summary.csv'), index=False)
display(HTML(linear_summary_df.to_html(index=False)))
print(f"Summary saved to {os.path.join(OUTPUT_DIR, 'linear_summary.csv')}")

### Visual Comparison of F1 Scores Across Scenarios

In [ ]:
# Pivot table for heatmap
f1_pivot = linear_summary_df.pivot(index='Scenario', columns='Method', values='F1 (mean)')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(f1_pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax)
ax.set_title('F1 Score (Variable Selection) - Higher is better')
plt.tight_layout()
plt.show()

# Pivot table for MSE
mse_pivot = linear_summary_df.pivot(index='Scenario', columns='Method', values='MSE')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(mse_pivot, annot=True, fmt='.3f', cmap='viridis_r', ax=ax)
ax.set_title('Out-of-sample MSE - Lower is better')
plt.tight_layout()
plt.show()

## Part 2: GLM Experiments (All Families)

Evaluate XLasso across all GLM families:
- Gaussian (linear regression)
- Binomial (logistic regression)
- Poisson (count regression)
- Multinomial (multi-class classification)
- Cox (proportional hazards survival analysis)

In [ ]:
# Run all GLM experiments
N_REPEATS_GLM = 5  # Change to 20 for full results

glm_results = run_all_glm_experiments(
    output_dir=OUTPUT_DIR,
    n_repeats_override=N_REPEATS_GLM,
    save_plots=True
)

print('\n=== GLM Experiments Summary ===')
for family, df in glm_results.items():
    print(f'\n**{family}**')
    if 'f1' in df.columns:
        display(df[['f1']].round(4))

### GLM Summary Table

In [ ]:
glm_summary_rows = []
for family, df in glm_results.items():
    for method in df.index:
        row = {
            'Family': family,
            'Method': method,
            'F1 (mean)': df.loc[method, ('f1', 'mean')] if ('f1', 'mean') in df.columns else np.nan,
            'F1 (std)': df.loc[method, ('f1', 'std')] if ('f1', 'std') in df.columns else np.nan,
        }
        # Add family-specific performance metric
        if family == 'gaussian' and ('mse', 'mean') in df.columns:
            row['Performance'] = df.loc[method, ('mse', 'mean')]
            row['Metric'] = 'MSE'
        elif family == 'binomial' and ('auc', 'mean') in df.columns:
            row['Performance'] = df.loc[method, ('auc', 'mean')]
            row['Metric'] = 'AUC'
        elif family == 'poisson' and ('deviance', 'mean') in df.columns:
            row['Performance'] = df.loc[method, ('deviance', 'mean')]
            row['Metric'] = 'Deviance'
        elif family == 'multinomial' and ('accuracy', 'mean') in df.columns:
            row['Performance'] = df.loc[method, ('accuracy', 'mean')]
            row['Metric'] = 'Accuracy'
        elif family == 'cox' and ('cindex', 'mean') in df.columns:
            row['Performance'] = df.loc[method, ('cindex', 'mean')]
            row['Metric'] = 'C-index'
        glm_summary_rows.append(row)

glm_summary_df = pd.DataFrame(glm_summary_rows)
glm_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'glm_summary.csv'), index=False)
display(HTML(glm_summary_df.to_html(index=False)))

## Part 3: Nonlinear Experiments

Two-dimensional comparison:
1. **Across univariate models**: linear vs spline vs tree
2. **Across XLasso configurations**: baseline vs adaptive vs grouping vs full

Data scenarios:
- pure nonlinear: all relevant features have nonlinear relationships
- mixed nonlinear: mix of linear, nonlinear, and irrelevant features
- block nonlinear: true nonlinear features are in correlated blocks
- null model: all features are irrelevant (tests FDR control)

In [ ]:
# Run all nonlinear experiments
N_REPEATS_NONLINEAR = 5  # Change to 20 for full results

nonlinear_results = run_all_nonlinear_experiments(
    output_dir=OUTPUT_DIR,
    n_repeats_override=N_REPEATS_NONLINEAR,
    save_plots=True
)

print('\n=== Nonlinear Experiments Summary ===')

### Nonlinear Summary Table

In [ ]:
nonlinear_summary_rows = []
for scenario, df in nonlinear_results.items():
    for method in df.index:
        univariate_type = method.split(' - ')[0]
        config = method.split(' - ')[1] if ' - ' in method else method
        row = {
            'Scenario': scenario,
            'Univariate': univariate_type,
            'Configuration': config,
            'Method': method,
            'F1 (mean)': df.loc[method, ('f1', 'mean')],
            'F1 (std)': df.loc[method, ('f1', 'std')],
            'MSE': df.loc[method, ('mse', 'mean')],
        }
        nonlinear_summary_rows.append(row)

nonlinear_summary_df = pd.DataFrame(nonlinear_summary_rows)
nonlinear_summary_df.to_csv(os.path.join(OUTPUT_DIR, 'nonlinear_summary.csv'), index=False)

display(HTML(nonlinear_summary_df.to_html(index=False)))

### Comparison of Univariate Models (Full XLasso)

In [ ]:
# Compare F1 across univariate models for full XLasso
full_nonlinear = nonlinear_summary_df[nonlinear_summary_df['Configuration'] == 'full']
if len(full_nonlinear) > 0:
    f1_pivot = full_nonlinear.pivot(index='Scenario', columns='Univariate', values='F1 (mean)')
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(f1_pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax)
    ax.set_title('F1 Score - Full XLasso by Univariate Model (Higher = better)')
    plt.tight_layout()
    plt.show()

    mse_pivot = full_nonlinear.pivot(index='Scenario', columns='Univariate', values='MSE')
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(mse_pivot, annot=True, fmt='.3f', cmap='viridis_r', ax=ax)
    ax.set_title('Out-of-sample MSE - Full XLasso by Univariate Model (Lower = better)')
    plt.tight_layout()
    plt.show()

### Effect of Grouping and Adaptive Weighting in Nonlinear Setting

In [ ]:
# Compare F1 across configurations for each univariate type (averaged across scenarios)
config_comparison = nonlinear_summary_df.groupby(['Univariate', 'Configuration'])['F1 (mean)'].mean().unstack()
display(config_comparison.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
config_comparison.plot(kind='bar', ax=ax)
ax.set_ylabel('Average F1 (higher is better)')
ax.set_title('F1 Comparison by Configuration - Nonlinear Experiments')
plt.xticks(rotation=0)
plt.legend(title='Configuration')
plt.tight_layout()
plt.show()

## Overall Summary

All results are saved in `../experiments/results/`:
- CSV files with aggregated results
- PNG and PDF figures
- Summary CSV tables for all three parts

Key research questions to answer from results:

1. **Linear**: Do group constraint and adaptive weighting improve variable selection accuracy in the presence of correlation?
2. **GLM**: Do the improvements generalize across all GLM families?
3. **Nonlinear**: Do nonlinear univariate models (spline/tree) improve prediction on truly nonlinear data? Does XLasso still help with variable selection?

## Generate LaTeX Tables for Paper


In [ ]:
# Generate LaTeX table for linear results
from experiments.visualization import results_to_latex_table

print("\n=== LaTeX: Linear F1 by scenario and method ===\n")
scenarios = linear_summary_df['Scenario'].unique().tolist()
methods = ['XLasso (baseline)', 'XLasso + adaptive', 'XLasso + grouping', 'XLasso (full)']
latex_linear_f1 = results_to_latex_table(
    linear_results if 'linear_results' in locals() else {},
    methods,
    scenarios,
    "F1 score for linear Gaussian experiments across scenarios",
    "tab:linear_f1"
)
print(latex_linear_f1)

with open(os.path.join(OUTPUT_DIR, 'linear_f1_table.tex'), 'w') as f:
    f.write(latex_linear_f1)
print(f"\nSaved to {os.path.join(OUTPUT_DIR, 'linear_f1_table.tex')}")